<a href="https://colab.research.google.com/github/KlyffHanger/TinyML/blob/main/Quantization_Aware_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview

Welcome to an end-to-end example for *quantization aware training*.

### Summary

In this tutorial, you will:

1.   Train a `keras` model for MNIST from scratch.
2.   Fine tune the model by applying the quantization aware training API, see the accuracy, and
     export a quantization aware model.
3.   Use the model to create an actually quantized model for the TFLite
     backend.
4.   See the persistence of accuracy in
     TFLite and a 4x smaller model. To see the latency benefits on mobile, try out the TFLite examples [in the TFLite app repository](https://www.tensorflow.org/lite/models).

## Setup

In [1]:
# Run this first. It will install tensorflow_model_optimization if missing.
# Colab usually already has TF; we install/upgrade tf-mot.
# Uninstall existing tensorflow and tensorflow-model-optimization
!pip uninstall -y tensorflow tensorflow-model-optimization
# Install tf-model-optimization (quiet)
!pip install -q -U tensorflow tensorflow-model_optimization
# (Optional) show versions
import tensorflow as tf, tensorflow_model_optimization as tfmot
print("TensorFlow:", tf.__version__)
print("TF-MOT:", tfmot.__version__)


Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
Found existing installation: tensorflow-model-optimization 0.8.0
Uninstalling tensorflow-model-optimization-0.8.0:
  Successfully uninstalled tensorflow-model-optimization-0.8.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.20.0 which is incompatible.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.20.0 which is incompatible.
tf-keras 2.19.0 requires tensorflow<2.20,>=2.19, but you have tensorflow 2.20.0 which is incompatible.
TensorFlow: 2.20.0
TF-MOT: 0.8.0


In [2]:

import os
import pathlib
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

try:
    import tensorflow_model_optimization as tfmot
except Exception as e:
    raise RuntimeError("Please install tensorflow_model_optimization. Run the install cell above.") from e

## Train a model for MNIST without quantization aware training

In [3]:
# Load MNIST dataset
mnist = tf.keras.datasets.mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# Normalize the input image so that each pixel value is between 0 to 1.
train_images = train_images / 255.0
test_images = test_images / 255.0

# Define the model architecture.
model = keras.Sequential([
  keras.layers.InputLayer(input_shape=(28, 28)),
  keras.layers.Reshape(target_shape=(28, 28, 1)),
  keras.layers.Conv2D(filters=12, kernel_size=(3, 3), activation='relu'),
  keras.layers.MaxPooling2D(pool_size=(2, 2)),
  keras.layers.Flatten(),
  keras.layers.Dense(10)
])

model.summary()

# Train the digit classification model
model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

model.fit(
  train_images,
  train_labels,
  epochs=2,
  validation_split=0.1,
)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 12)        120       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 12)        0         
 D)                                                              
                                                                 
 flatten (Flatten)           (None, 2028)              0         
                                                                 
 dense (Dense)               (None, 10)                20290     
                                                                 
Total params: 20410 (79.73 KB)
Trainable params: 20410 (79.73 KB)
Non-t

### Let us save the REGULAR model and get the size


In [4]:
REG_MODEL_SAVED_MODEL_DIR = "reg_saved_model"
model.export(REG_MODEL_SAVED_MODEL_DIR)

import os

reg_model_size = sum(os.path.getsize(os.path.join(dirpath, filename)) for dirpath, dirnames, filenames in os.walk(REG_MODEL_SAVED_MODEL_DIR) for filename in filenames)
print(f"Regular Model directory size: {reg_model_size / (1024):.2f} KB")

Saved artifact at 'reg_saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28), dtype=tf.float32, name='input_1')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  131922618575824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131922618577552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131922618576976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131922618578128: TensorSpec(shape=(), dtype=tf.resource, name=None)
Regular Model directory size: 105.45 KB


## Clone and fine-tune pre-trained model with quantization aware training


### Define the model

You will apply quantization aware training to the whole model and see this in the model summary. All layers are now prefixed by "quant".

Note that the resulting model is quantization aware but not quantized (e.g. the weights are float32 instead of int8). The sections after show how to create a quantized model from the quantization aware one.

In the [comprehensive guide](https://www.tensorflow.org/model_optimization/guide/quantization/training_comprehensive_guide.md), you can see how to quantize some layers for model accuracy improvements.

In [5]:
# !pip install tensorflow-model-optimization
import tensorflow_model_optimization as tfmot

quantize_model = tfmot.quantization.keras.quantize_model

# q_aware stands for for quantization aware.
q_aware_model = quantize_model(model)

# `quantize_model` requires a recompile.
q_aware_model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

q_aware_model.summary()

q_aware_model.fit(
  train_images,
  train_labels,
  epochs=2,
  validation_split=0.1,
)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 quantize_layer (QuantizeLa  (None, 28, 28)            3         
 yer)                                                            
                                                                 
 quant_reshape (QuantizeWra  (None, 28, 28, 1)         1         
 pperV2)                                                         
                                                                 
 quant_conv2d (QuantizeWrap  (None, 26, 26, 12)        147       
 perV2)                                                          
                                                                 
 quant_max_pooling2d (Quant  (None, 13, 13, 12)        1         
 izeWrapperV2)                                                   
                                                                 
 quant_flatten (QuantizeWra  (None, 2028)              1

### Let us save the QUANTIZED AWARE model and get the size


In [6]:
QUANTIZED_AWARE_MODEL_SAVED_MODEL_DIR = "quantized_aware_saved_model"
model.export(QUANTIZED_AWARE_MODEL_SAVED_MODEL_DIR)

import os

quant_aware_model_size = sum(os.path.getsize(os.path.join(dirpath, filename)) for dirpath, dirnames, filenames in os.walk(QUANTIZED_AWARE_MODEL_SAVED_MODEL_DIR) for filename in filenames)
print(f"Quantized Aware Model directory size: {quant_aware_model_size / (1024):.2f} KB")

Saved artifact at 'quantized_aware_saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28), dtype=tf.float32, name='input_1')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  131922618575824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131922618577552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131922618576976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131922618578128: TensorSpec(shape=(), dtype=tf.resource, name=None)
Quantized Aware Model directory size: 105.45 KB


### Train and evaluate the model against baseline

To demonstrate fine tuning after training the model for just an epoch, fine tune with quantization aware training on a subset of the training data.

In [7]:
train_images_subset = train_images[0:1000] # out of 60000
train_labels_subset = train_labels[0:1000]

q_aware_model.fit(train_images_subset, train_labels_subset,
                  batch_size=500, epochs=1, validation_split=0.1)

2/2 [==============================] - 0s 162ms/step - loss: 0.0828 - accuracy: 0.9778 - val_loss: 0.0854 - val_accuracy: 0.9800


For this example, there is minimal to no loss in test accuracy after quantization aware training, compared to the baseline.

In [8]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

_, q_aware_model_accuracy = q_aware_model.evaluate(
   test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Quant test accuracy:', q_aware_model_accuracy)

Baseline test accuracy: 0.9664999842643738
Quant test accuracy: 0.9772999882698059


## Create quantized model for TFLite backend

After this, you have an actually quantized model with int8 weights and uint8 activations.

In [9]:
# make TFLite model from Regular Model
converter1 = tf.lite.TFLiteConverter.from_keras_model(model)
converter1.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter1.convert()

# make TFLite model from Quantization Aware Model
converter2 = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
converter2.optimizations = [tf.lite.Optimize.DEFAULT]

quantized_tflite_model = converter2.convert()

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


### Let us get the size of the converted regular model and quantization aware model

In [10]:
import pathlib

tflite_models_dir = pathlib.Path("/tmp/")

tflite_model_file = tflite_models_dir/'regular.tflite'
tflite_model_file.write_bytes(tflite_model)

quant_aware_model_file = tflite_models_dir/'quantized_aware.tflite'
quant_aware_model_file.write_bytes(quantized_tflite_model)

tflite_model_size = os.path.getsize(tflite_models_dir/'regular.tflite')
print(f"TFLite model file size: {tflite_model_size / (1024):.2f} KB")

quant_aware_model_size = os.path.getsize(tflite_models_dir/'quantized_aware.tflite')
print(f"Quantized Aware TFLite model file size: {quant_aware_model_size/ (1024):.2f} KB")

TFLite model file size: 23.41 KB
Quantized Aware TFLite model file size: 24.18 KB


## See persistence of accuracy from TF to TFLite

Define a helper function to evaluate the TF Lite model on the test dataset.

In [11]:
import numpy as np

def evaluate_model(interpreter):
  input_index = interpreter.get_input_details()[0]["index"]
  output_index = interpreter.get_output_details()[0]["index"]

  # Run predictions on every image in the "test" dataset.
  prediction_digits = []
  for i, test_image in enumerate(test_images):
    if i % 1000 == 0:
      print('Evaluated on {n} results so far.'.format(n=i))
    # Pre-processing: add batch dimension and convert to float32 to match with
    # the model's input data format.
    test_image = np.expand_dims(test_image, axis=0).astype(np.float32)
    interpreter.set_tensor(input_index, test_image)

    # Run inference.
    interpreter.invoke()

    # Post-processing: remove batch dimension and find the digit with highest
    # probability.
    output = interpreter.tensor(output_index)
    digit = np.argmax(output()[0])
    prediction_digits.append(digit)

  print('\n')
  # Compare prediction results with ground truth labels to calculate accuracy.
  prediction_digits = np.array(prediction_digits)
  accuracy = (prediction_digits == test_labels).mean()
  return accuracy

You evaluate the quantized model and see that the accuracy from TensorFlow persists to the TFLite backend.

In [12]:
# !pip install ai-edge-litert
# from ai_edge_litert.interpreter import Interpreter

#interpret with Regular TFLite model
interpreter1 = tf.lite.Interpreter(model_content=tflite_model)
# interpreter1 = Interpreter(model_content=tflite_model)
interpreter1.allocate_tensors()
test_accuracy1 = evaluate_model(interpreter1)


#interpret with Quantization Aware TFLite model
interpreter2 = tf.lite.Interpreter(model_content=quantized_tflite_model)
# interpreter2 = Interpreter(model_content=quantized_tflite_model)
interpreter2.allocate_tensors()
test_accuracy2 = evaluate_model(interpreter2)


print('Regular TF test accuracy:', baseline_model_accuracy)
print('Regular TFLite test_accuracy:', test_accuracy1)

print('Quant TF test accuracy:', q_aware_model_accuracy)
print('Quant TFLite test_accuracy:', test_accuracy2)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Evaluated on 0 results so far.
Evaluated on 1000 results so far.
Evaluated on 2000 results so far.
Evaluated on 3000 results so far.
Evaluated on 4000 results so far.
Evaluated on 5000 results so far.
Evaluated on 6000 results so far.
Evaluated on 7000 results so far.
Evaluated on 8000 results so far.
Evaluated on 9000 results so far.


Evaluated on 0 results so far.
Evaluated on 1000 results so far.
Evaluated on 2000 results so far.
Evaluated on 3000 results so far.
Evaluated on 4000 results so far.
Evaluated on 5000 results so far.
Evaluated on 6000 results so far.
Evaluated on 7000 results so far.
Evaluated on 8000 results so far.
Evaluated on 9000 results so far.


Regular TF test accuracy: 0.9664999842643738
Regular TFLite test_accuracy: 0.9668
Quant TF test accuracy: 0.9772999882698059
Quant TFLite test_accuracy: 0.9773


## See 4x smaller model from quantization

You create a float TFLite model and then see that the quantized TFLite model
is 4x smaller.

In [13]:
import tempfile

# Create float TFLite model.
float_converter = tf.lite.TFLiteConverter.from_keras_model(model)
float_tflite_model = float_converter.convert()

# Measure sizes of models.
_, float_file = tempfile.mkstemp('.tflite')
_, quant_file = tempfile.mkstemp('.tflite')

with open(quant_file, 'wb') as f:
  f.write(quantized_tflite_model)

with open(float_file, 'wb') as f:
  f.write(float_tflite_model)

print("Float model in Mb:", os.path.getsize(float_file) / float(2**20))
print("Quantized model in Mb:", os.path.getsize(quant_file) / float(2**20))

Float model in Mb: 0.08069610595703125
Quantized model in Mb: 0.02361297607421875
